# Instalación y Librerías

In [2]:
# pip install ollama

In [28]:
# Para tratamiento de datos
import pandas as pd
import numpy as np
import re #para llamar a Expresiones Regulares y estandarizar el nombre de las columnas.
import time
import requests

# Para visualización de datos
import matplotlib.pyplot as plt
import seaborn as sns

# Para poder visualizar todas las columnas de los DataFrames
pd.set_option('display.max_columns', None) 

# Trabajar con el sistema operativo y variables de entorno
import os 
from dotenv import load_dotenv
load_dotenv() #carga las variables del entorno .env; devuelve un true o false
import ollama
MODELO = 'gemma3:4b' 

# Conexión con MySQL
import mysql.connector
from mysql.connector import Error

# Gestión de los warnings
import warnings
warnings.filterwarnings("ignore")

#Gestión de las descargas
from tqdm import tqdm #barra progreso
from concurrent.futures import ThreadPoolExecutor, as_completed #paralelismo

MAX_THREADS_STEAM = 3   # Máximo 2-3 para evitar baneos de Steam
MAX_THREADS_OLLAMA = 4  # Ajusta según tu GPU/RAM (4-8 suele ser ideal)
PAUSA_STEAM = 1.0       # Respiro extra entre peticiones concurrentes

In [ ]:
# Cargamos las variables de entorno del archivo .env
load_dotenv()

# 1. Recuperamos la Key y el ID base
steam_key = os.getenv("steam_key")

# 2. Creamos la lista de los 16 IDs dinámicamente
lista_ids = [76561198056243081, 76561198842603734, 76561198023414915, 76561198048165534, 76561198254085126, 76561197984432884, 76561199080934614, 76561198294650349, 
             76561197986603983, 76561198212206651, 76561198062673538, 76561198014898339, 76561198067053149, 76561199499421434, 76561198046160451]

print(f"Se han cargado {len(lista_ids)} IDs.")

Se han cargado 15 IDs desde el archivo .env


# DS Jugadores

In [ ]:
steam_key = os.getenv("steam_key")

# Cargamos los IDs desde el .env
ids_proyecto = [os.getenv(f"steam_id_{i}") for i in range(16)]
ids_proyecto = [i for i in ids_proyecto if i] # Limpiamos valores None

registros_juegos = []

print(f"--- Iniciando construcción del Dataset ({len(ids_proyecto)} usuarios) ---")

# --- 2. EXTRACCIÓN DE JUEGOS DE LA API (Manteniendo tu lógica original) ---
for s_id in ids_proyecto:
    url = "https://api.steampowered.com/IPlayerService/GetOwnedGames/v0001/"
    params = {
        'key': steam_key,
        'steamid': s_id,
        'format': 'json',
        'include_appinfo': 1,
        'include_played_free_games': 1
    }
    
    try:
        response = requests.get(url, params=params, timeout=10)
        
        if response.status_code == 200:
            data = response.json()
            juegos = data.get('response', {}).get('games', [])
            
            for juego in juegos:
                juego['user_id'] = s_id
                registros_juegos.append(juego)
            
            print(f"✅ ID {s_id}: {len(juegos)} juegos añadidos.")
        else:
            print(f"❌ ID {s_id}: Error HTTP {response.status_code}")
            
    except Exception as e:
        print(f"⚠️ ID {s_id}: Error: {e}")
    
    time.sleep(0.5)

# --- 3. PROCESAMIENTO Y MUESTREO (25k registros) ---
if registros_juegos:
    df_steam = pd.DataFrame(registros_juegos)
    print(f"\nTotal de registros extraídos: {len(df_steam)}")

    # Calculamos cuántas filas tomar (máximo 25,000)
    total_a_muestrear = min(25000, len(df_steam))
    
    # Muestreo estratificado: Asegura registros de TODOS los usuarios
    # Se toma una muestra proporcional de cada usuario y luego se mezcla
    df_sample = df_steam.groupby('user_id', group_keys=False).apply(
        lambda x: x.sample(min(len(x), total_a_muestrear // len(ids_proyecto)))
    ).sample(frac=1).reset_index(drop=True)

    print(f"Muestra seleccionada para enriquecer: {len(df_sample)} registros.")
    
    # --- 4. ENRIQUECIMIENTO CON LOGROS ---
    def obtener_estadisticas_logros(api_key, steam_id, app_id):
        url = "https://api.steampowered.com/ISteamUserStats/GetPlayerAchievements/v0001/"
        params = {'key': api_key, 'steamid': steam_id, 'appid': app_id}
        try:
            r = requests.get(url, params=params, timeout=5)
            if r.status_code == 200:
                data = r.json()
                achievements = data.get('playerstats', {}).get('achievements', [])
                total = len(achievements)
                ganados = sum(1 for a in achievements if a.get('achieved') == 1)
                return ganados, total
        except:
            return 0, 0
        return 0, 0

    logros_data = []
    print("Enriqueciendo datos con logros... (esto tardará bastante)")

    # Usamos enumerate para el contador real del progreso
    for i, (idx, row) in enumerate(df_sample.iterrows()):
        ganados, totales = obtener_estadisticas_logros(steam_key, row['user_id'], row['appid'])
        
        logros_data.append({
            'achieved': ganados,
            'total_achievements': totales,
            'completion_percentage': (ganados / totales * 100) if totales > 0 else 0
        })

        # Contador de progreso real cada 100 registros
        if i % 5000 == 0:
            print(f"Procesados {i} de {len(df_sample)}...")
        
        time.sleep(0.1) # Pausa de cortesía para la API

    # --- 5. UNIR Y GUARDAR FINAL ---
    df_extra = pd.DataFrame(logros_data)
    df_final = pd.concat([df_sample.reset_index(drop=True), df_extra], axis=1)

    # Limpieza
    df_final = df_final.drop(columns=['img_icon_url', 'has_leaderboards'], errors='ignore')
    df_final["content_descriptorids"] = df_final["content_descriptorids"].fillna(0)
    
    def es_adulto(valor):
    # Si el valor es 0, es False. Si es cualquier otra cosa (lista o ID), es True.
        if valor == 0:
            return False
        else:
            return True

    # Aplicamos la lógica a la columna
    df_final['adult_descriptorids'] = df_final['content_descriptorids'].apply(es_adulto)
    

    # Guardado con el separador "||"
    df_final.to_csv("./datasets/dataset_steam.csv", index=False)
    print(f"¡Hecho! Dataset actualizado con {len(df_final)} filas.")
else:
    print("\nNo se pudo extraer información. Revisa tu steam_key.")

--- Iniciando construcción del Dataset (16 usuarios) ---
✅ ID 76561198056243081: 41 juegos añadidos.
✅ ID 76561198842603734: 28623 juegos añadidos.
✅ ID 76561198023414915: 4646 juegos añadidos.
✅ ID 76561199080934614: 23328 juegos añadidos.
✅ ID 76561197984432884: 3043 juegos añadidos.
✅ ID 76561198254085126: 67 juegos añadidos.
✅ ID 76561198048165534: 15682 juegos añadidos.
✅ ID 76561198023455525: 8895 juegos añadidos.
✅ ID 76561198294650349: 1345 juegos añadidos.
✅ ID 76561197986603983: 14690 juegos añadidos.
✅ ID 76561198212206651: 2817 juegos añadidos.
✅ ID 76561198062673538: 1272 juegos añadidos.
✅ ID 76561198014898339: 3581 juegos añadidos.
✅ ID 76561198067053149: 3336 juegos añadidos.
✅ ID 76561199499421434: 4251 juegos añadidos.
✅ ID 76561198046160451: 1315 juegos añadidos.

Total de registros extraídos: 116932
Muestra seleccionada para enriquecer: 21222 registros.
Enriqueciendo datos con logros... (esto tardará bastante)
Procesados 0 de 21222...
Procesados 5000 de 21222...
Pro

# DS Contenido +18

In [ ]:
df_final = pd.read_csv("./datasets/dataset_steam.csv")

data_mapeo_contenido = {
    'content_id': [0, 1, 2, 3, 4, 5],
    'description_es': [
        'Contenido family friendly',
        'Contenido sexual explícito o desnudez',
        'Violencia frecuente o intensa',
        'Lenguaje fuerte',
        'Temas adultos/sensibles',
        'Escenas de sangre y desmembramiento'
    ]
}

# 2. Creamos el DataFrame de mapeo
df_mapeo = pd.DataFrame(data_mapeo_contenido)

# 3. Extraemos los IDs únicos de la columna con listas
# Usamos explode para separar las listas en filas individuales y luego unique()
# Filtrando primero los nulos para evitar errores.
unique_ids = pd.Series(
    [item for sublist in df_final['content_descriptorids'].dropna() for item in sublist]
).unique()

# 4. Filtramos el df_mapeo para tener solo los que aparecen en df_final
df_descriptors_unicos = df_mapeo[df_mapeo['content_id'].isin(unique_ids)].reset_index(drop=True)
df_descriptors_unicos.to_csv("./datasets/mapeo_contenido.csv", index=False)


# DS Maestro de juegos

In [ ]:
# --- CONFIGURACIÓN ---
MODELO = "gemma3:4b"  # Recomendado por velocidad
ARCHIVO_ENTRADA = "./datasets/dataset_steam.csv"
ARCHIVO_MAESTRO = "./datasets/juegos_maestro.csv"


MAX_THREADS_STEAM = 3   # Máximo 2-3 para evitar baneos de Steam
MAX_THREADS_OLLAMA = 4  # Ajusta según tu GPU/RAM (4-8 suele ser ideal)
PAUSA_STEAM = 1.0       # Respiro extra entre peticiones concurrentes

if not os.path.exists('./datasets'): os.makedirs('./datasets')

# --- FUNCIONES DE APOYO ---

def obtener_info_juego(appid):
    url = f"https://store.steampowered.com/api/appdetails?appids={appid}&l=spanish"
    try:
        response = requests.get(url, timeout=10)
        if response.status_code == 429:
            time.sleep(20)
            return None
        data = response.json()
        if data and data.get(str(appid), {}).get('success'):
            details = data[str(appid)]['data']
            genres_list = details.get('genres', [])
            return {
                'appid': appid,
                'name': details.get('name', 'N/A'),
                'developer': details.get('developers', ["Desconocido"])[0],
                'genres_steam': ", ".join([g['description'] for g in genres_list]) if genres_list else "Sin Género",
                'price': 0 if details.get('is_free') else (details.get('price_overview', {}).get('final', 0) / 100),
                'country': None,
                'genre': None,
                'release_year': int(pd.to_datetime(date_str, errors='coerce').year),
                'metacritic_score': details.get('metacritic', {}).get('score', 0),
                'total_recommendations': details.get('recommendations', {}).get('total', 0),
                'is_free': details.get('is_free', False)
            }
    except: return None

def obtener_pais_worker(dev):
    """Worker para procesar países en paralelo."""
    prompt = f"""Empresa: {dev}. Eres un buscador de sedes corporativas.
    Tarea: Indica el país donde se encuentra la sede principal de la empresa: '{dev}'.
    
    Reglas críticas de respuesta:
    1. Responde ÚNICAMENTE con el nombre del país en español.
    2. NO escribas frases como "La sede está en..." o "El país es...".
    3. NO uses puntos finales ni signos de puntuación.
    4. NO des ninguna información adicional, ni ciudades, ni fechas.
    5. Si no conoces el país, responde exclusivamente con la palabra: Desconocido.
    6. Responde con exactamente UNA palabra.
    """
    try:
        response = ollama.chat(model=MODELO, messages=[{'role': 'user', 'content': prompt}])
        pais = response['message']['content'].strip().split('\n')[0].replace(".", "")
        return dev, pais
    except: return dev, "Error"

def obtener_genero_worker(combo_steam):
    """Clasifica un combo de etiquetas Steam en un género genérico."""
    generos_referencia = "Acción, RPG, Estrategia, Aventura, Simulación, Deportes, Puzzle, Shooter"
    
    # FIX: el parámetro es un combo de géneros Steam, no un nombre de juego
    prompt = f"""
    Eres un experto clasificador de videojuegos.
    Objetivo: A partir de estas etiquetas de Steam '{combo_steam}', determina el género principal más genérico.
    Reglas:
    1. Responde con exactamente UNA palabra.
    2. Género genérico, NO específico.
    3. Usa esta lista como preferencia: [{generos_referencia}].
    4. NO uses puntos ni explicaciones.
    """

    try:
        response = ollama.chat(model=MODELO, messages=[{'role': 'user', 'content': prompt}])
        genero = response['message']['content'].strip().split('\n')[0].replace(".", "").capitalize()
        return combo_steam, genero
    except Exception as e:
        return combo_steam, "Error"

# --- PROCESO PRINCIPAL ---

# 1. Cargar o Crear Dataset Maestro
if os.path.exists(ARCHIVO_MAESTRO):
    df_maestro = pd.read_csv(ARCHIVO_MAESTRO)
else:
    df_maestro = pd.DataFrame(columns=['appid', 'name', 'developer', 'genres_steam', 'price', 'country', 'genre', 'release_year', 'metacritic_score', 'total_recommendations', 'is_free' ])

# PASO 1: Steam
df_inicial = pd.read_csv(ARCHIVO_ENTRADA)
ids_totales = df_inicial["appid"].astype(int).unique()
ids_maestro = set(df_maestro['appid'].astype(int).values)  # set() hace la búsqueda O(1)

ids_faltantes = [aid for aid in ids_totales if aid not in ids_maestro]

if ids_faltantes:
    print(f"--- PASO 1: Steam ({len(ids_faltantes)} juegos, {MAX_THREADS_STEAM} hilos) ---")
    with ThreadPoolExecutor(max_workers=MAX_THREADS_STEAM) as executor:
        futures = {executor.submit(obtener_info_juego, aid): aid for aid in ids_faltantes}
        nuevos_registros = []
        for f in tqdm(as_completed(futures), total=len(futures), desc="Steam", bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt}"):
            res = f.result()
            if res:
                nuevos_registros.append(res)
            
            # Guardado incremental cada 20 resultados para no perder nada
            if len(nuevos_registros) >= 20:
                df_maestro = pd.concat([df_maestro, pd.DataFrame(nuevos_registros)], ignore_index=True)
                df_maestro.to_csv(ARCHIVO_MAESTRO, index=False)
                nuevos_registros = []
            time.sleep(PAUSA_STEAM / MAX_THREADS_STEAM)
        
        if nuevos_registros:
            df_maestro = pd.concat([df_maestro, pd.DataFrame(nuevos_registros)], ignore_index=True)
            df_maestro.to_csv(ARCHIVO_MAESTRO, index=False)

df_maestro.drop(columns=['country', 'genre'], inplace=True, errors='ignore')

# Llamada a Ollama para los 
print(f"\n--- PASO 2: Ollama Países ---")

devs_totales = df_maestro['developer'].unique()
print(f"Total de desarrolladores a procesar: {len(devs_totales)}")

with ThreadPoolExecutor(max_workers=MAX_THREADS_OLLAMA) as executor:
    futures = [executor.submit(obtener_pais_worker, d) for d in devs_totales]
    
    contador_pais = 0  # FIX: contador incremental real
    for f in tqdm(as_completed(futures), total=len(futures), desc="Países", bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt}"):
        dev, pais = f.result()
        df_maestro.loc[df_maestro['developer'] == dev, 'country'] = pais
        
        contador_pais += 1
        if contador_pais % 20 == 0:  # FIX: ahora sí guarda cada 20 devs procesados
            df_maestro.to_csv(ARCHIVO_MAESTRO, index=False)

# Guardado final Paso 2
df_maestro.to_csv(ARCHIVO_MAESTRO, index=False)


print(f"\n--- PASO 3: Ollama Géneros ---")

combos_totales = df_maestro['genres_steam'].unique()
print(f"Total de combos únicos a reclasificar: {len(combos_totales)}")

with ThreadPoolExecutor(max_workers=MAX_THREADS_OLLAMA) as executor:
    futures = {executor.submit(obtener_genero_worker, c): c for c in combos_totales}
    
    contador_genero = 0
    for f in tqdm(as_completed(futures), total=len(futures), desc="Géneros", bar_format="{l_bar}{bar}| {n_fmt}/{total_fmt}"):
        combo_original = futures[f]
        try:
            _, genero_limpio = f.result()
            df_maestro.loc[df_maestro['genres_steam'] == combo_original, 'genre'] = genero_limpio
            
            contador_genero += 1
            if contador_genero % 15 == 0:
                df_maestro.to_csv(ARCHIVO_MAESTRO, index=False)
        except Exception as e:
            print(f"Error en combo {combo_original}: {e}")

# Guardado final absoluto
df_maestro.to_csv(ARCHIVO_MAESTRO, index=False)
print(f"\n✅ Proceso completo. Datos en {ARCHIVO_MAESTRO}")

Extrayendo datos de Steam (12408 juegos)...
Procesando localizaciones con Ollama...
Procesando géneros con Ollama...

Procesamiento completo. Dataset final:


,appid,name,developer,price,is_free,metacritic_score,total_recommendations,country,genre
0,22380,Fallout: New Vegas,Obsidian Entertainment,9.99,False,84,204108,Estados Unidos,RPG
1,39560,Painkiller: Resurrection,Homegrown Games,9.99,False,38,510,Reino Unido,Acción
2,434750,Z-Exemplar,Suminell Studios,2.39,False,0,0,España,RPG
3,1036440,Kingdom Wars 2: Definitive Edition,Reverie World Studios,3.79,False,0,341,Hong Kong,Estrategia
4,876930,Fancy Skiing 2: Online,哈视奇科技,6.99,False,0,0,China,Acción
